In [1]:
#import libraries
from astropy.io import fits
from astropy.wcs import WCS
from photutils.aperture import EllipticalAperture
from photutils.aperture import EllipticalAnnulus
from photutils.aperture import aperture_photometry
from astropy.coordinates import SkyCoord
import astropy.units as u
import pyregion
import numpy as np
from photutils.centroids import centroid_com
# from skimage.measure import regionprops, label
from astropy.nddata import Cutout2D
import matplotlib.pyplot as plt
#import statmorph
from photutils.segmentation import detect_sources
from astropy.visualization import ZScaleInterval, ImageNormalize

from matplotlib.patches import Ellipse
from astropy.visualization import simple_norm
from astropy.stats import SigmaClip
from photutils.background import Background2D, MedianBackground

from reproject import reproject_interp
import astroalign as aa

import pandas as pd

from matplotlib.backends.backend_pdf import PdfPages

import warnings
warnings.simplefilter('ignore')

import Galaxy_info as gi
import utils

In [2]:
#file data
file = gi.Galaxy_Info["IC 10"]

#galaxy name
galaxy = file[0]

#hi file
hi_fits = file[1]

#ha file
ha_fits = file[2]

#ha csv file
ha_csv = file[3]

#irac file
irac_fits = file[4]

#irac csv file
irac_csv = file[5]

#region file
region_file = file[6]

#distance to galxy kpc
distance = file[7]

In [9]:
# Load images

#hi image
hdu_hi = fits.open(hi_fits)
header_hi = hdu_hi[0].header
hi_image = hdu_hi[0].data
wcs_hi = WCS(header_hi)
cdelt = "CD2_2" if "CD2_2" in header_hi else "CDELT2"
arcsec_per_pixel_hi = np.abs(header_hi[cdelt]) * 3600

#ha image
hdu_ha2 = fits.open("/Users/tylerdonahue/Downloads/ic10hmrms.fits")
original_hdu = fits.open(ha_fits)
original_header = original_hdu[0].header
header_ha = original_header
correct = original_hdu[0].data #- utils.bkg_noise_offset(original_hdu[0].data)
ha_image, footprint = aa.register(correct.view(correct.dtype.newbyteorder('<')), hdu_ha2[0].data.view(hdu_ha2[0].data.dtype.newbyteorder('<')), detection_sigma=3.0)
wcs_ha = WCS(original_header)
cdelt = "CD2_2" if "CD2_2" in header_ha else "CDELT2"
arcsec_per_pixel_ha = np.abs(header_ha[cdelt]) * 3600


# hdu_ha2 = fits.open("/Users/tylerdonahue/Downloads/new-image.fits")
# original_hdu = fits.open(ha_fits)
# original_header = original_hdu[0].header
# hdu_ha, footprint = reproject_interp(hdu_ha2[0], original_header)
# header_ha = original_header#hdu_ha[0].header
# ha_image = hdu_ha#hdu_ha[0].data
# wcs_ha = WCS(original_header)
# cdelt = "CD2_2" if "CD2_2" in header_ha else "CDELT2"
# arcsec_per_pixel_ha = np.abs(header_ha[cdelt]) * 3600
#unit_ha = (10**header_ha["FLUX_CAL"]) / 1539

#irac 8 micron image
hdu_irac = fits.open(irac_fits)
header_irac = hdu_irac[0].header
irac_image = hdu_irac[0].data
wcs_irac = WCS(header_irac)
cdelt = "CD2_2" if "CD2_2" in header_irac else "CDELT2"
arcsec_per_pixel_irac = np.abs(header_irac[cdelt]) * 3600

ValueError: data must be finite, check for nan or inf values

In [ ]:
#load csvs

#load ha csv
ha_df = pd.read_csv(ha_csv)

#load irac csv
irac_df = pd.read_csv(irac_csv)

In [ ]:
#header_ha

In [ ]:
import utils

In [ ]:
original_header

In [ ]:
#load regions
regions = pyregion.open(region_file)

#initialize zscale
zscale = ZScaleInterval()

#open or create pdf
#with PdfPages(f"{galaxy}.pdf") as pdf:
if True:

    #iterate through regions
    for index, region in enumerate(regions):
    
        #check if applicable region
        if region.name != "ellipse":
            continue
    
        #grab coords of region center
        #coord_list: [RA, Dec, a_arcsec, b_arcsec, PA]
        ra, dec, a_arcsec, b_arcsec, pa_deg = region.coord_list
    
        #convert region center to pixel
        skycoord = SkyCoord(ra=ra*u.deg, dec=dec*u.deg)
        x_hi, y_hi = skycoord.to_pixel(wcs_hi)
        x_ha, y_ha = skycoord.to_pixel(wcs_ha)
        x_irac, y_irac = skycoord.to_pixel(wcs_irac)
    
        #convert to rads
        theta = np.deg2rad(pa_deg)
        print(ra, dec, pa_deg)
    
        #convert hi arcsec → pixels
        a_pix_hi = a_arcsec * 3600/ arcsec_per_pixel_hi 
        b_pix_hi = b_arcsec * 3600/ arcsec_per_pixel_hi
    
        #convert ha arcsec → pixels
        a_pix_ha = a_arcsec * 3600/ arcsec_per_pixel_ha
        b_pix_ha = b_arcsec * 3600/ arcsec_per_pixel_ha
    
        #convert irac arcsec → pixels
        a_pix_irac = a_arcsec * 3600/ arcsec_per_pixel_irac
        b_pix_irac = b_arcsec * 3600/ arcsec_per_pixel_irac

        #initialize image figure
        fig, axs = plt.subplots(1, 4, figsize=(20, 5))

        #set default values
        radii_hi = radii_ha = radii_irac = 0
        flux_hi = flux_ha = flux_irac = 0

        ##########################################################
        ### HI ###################################################
        ##########################################################

        #hi cutout subplot
        ax = axs[0]

        try:
            #calculate hi flux
            radii_hi, flux_hi, rim_core_hi = utils.sum_core_to_rim_pix((x_hi, y_hi), a_pix_hi/b_pix_hi, theta, hi_image.squeeze(), \
                                                     arcsec_per_pixel_hi, a_pix_hi)

            #convert radii to kpc
            radii_hi = radii_hi * distance * 4.85 * (10**-6)
        
        except Exception as e:
            
            #show exception
            print("HI", e)
        
        #check if valid coords in hi
        if(x_hi>0 and y_hi>0):
    
            #get hi hole type
            hi_hole_type = ha_df["HI Hole Type"][index]

            #remove empty dimensions from hi data
            data_hi = hi_image.squeeze()

            #cut image and define region ellipse
            cutout_hi, core_ellipse = utils.cut_mark_image((x_hi, y_hi), a_pix_hi, b_pix_hi, theta, data_hi)
            
            #check if cutout exists
            if type(cutout_hi) == "NoneType":
                continue
    
            #plot hi cut and region
            vmin, vmax = zscale.get_limits(cutout_hi)
            ax.imshow(cutout_hi, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)

            #plot region ellipse
            ax.add_patch(core_ellipse)

        #set hi title
        ax.set_title(f"Region: {index + 1} HI - Type {hi_hole_type}")
        plt.tight_layout()

        ##########################################################
        ### H-alpha ##############################################
        ##########################################################

        #initialize ha cutout subplot
        ax = axs[1]
        
        #get classification
        ha_class = ha_df["Classification"][index]

        #check if coverage
        if ha_class != "No Coverage" and ha_class != "No Emission":

            try:
                #calculate ha flux
                radii_ha, flux_ha, rim_core_ha = utils.sum_core_to_rim_pix((x_ha, y_ha), a_pix_ha/b_pix_ha, theta, ha_image, \
                                                         arcsec_per_pixel_ha, a_pix_ha)
        
                
                #convert radii to kpc
                radii_ha = radii_ha * distance * 4.85 * (10**-6)
            
            except Exception as e:
               
                #show exception
                print("Ha", e)
        
            #check if valid coords in ha
            if (x_ha>0 and y_ha>0):
    
                #cut image and define region ellipse
                data_ha = ha_image
                cutout_ha, core_ellipse = utils.cut_mark_image((x_ha, y_ha), a_pix_ha, b_pix_ha, theta, data_ha)
                
                #check id cutout exists
                if type(cutout_ha) == "NoneType":
                    continue
        
                #plot ha cut
                vmin, vmax = zscale.get_limits(cutout_ha)
                ax.imshow(cutout_ha, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
                
                #plot region ellipse
                ax.add_patch(core_ellipse)

        else:
            #hide blank axis
            ax.set_axis_off()

        #set ha title
        ax.set_title(f"H-alpha - Classification: {ha_class}")
        plt.tight_layout()

        ##########################################################
        ### IRAC 8 micron ########################################
        ##########################################################

        #initialize irac flux subplot
        ax = axs[2]

        #get classification
        irac_class = irac_df["Classification"][index]

        #check if coverage
        if irac_class != "No Coverage" and irac_class != "No Emission":

            try:
                #calculate irac flux
                radii_irac, flux_irac, rim_core_irac = utils.sum_core_to_rim_pix((x_irac, y_irac), a_pix_irac/b_pix_irac, theta, irac_image, \
                                                         arcsec_per_pixel_irac, a_pix_irac)
        
                #convert radii to Mpc
                radii_irac = radii_irac * distance * 4.85 * (10**-6)
            
            except Exception as e:
                
                #show exception
                print("IRAC", e)

            #check if valid coords in irac
            if(x_irac>0 and y_irac>0):
        
                #cut image and define region ellipse
                data_irac = irac_image
                cutout_irac, core_ellipse = utils.cut_mark_image((x_irac, y_irac), a_pix_irac, b_pix_irac, theta, data_irac)
                
                #check if cutout exists
                if type(cutout_irac) == "NoneType":
                    continue
        
                #plot irac cut
                vmin, vmax = zscale.get_limits(cutout_irac)
                ax.imshow(cutout_irac, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
                
                #plot region ellipse
                ax.add_patch(core_ellipse)

        else:
            #hide blank axis
            ax.set_axis_off()
        
        #set irac title
        ax.set_title(f"IRAC 8 micron - Classification: {irac_class}")
        plt.tight_layout()
        
        #save plots
        #pdf.savefig()

        #plot hi flux against ha flux
        if True:#ha_class != "No Coverage" and ha_class != "No Emission":

            if True:#ha_class == "Bubble-Like" or ha_class == "Diffuse":

                #plot core to rim sums for hi, ha, irac
                utils.plot_core_rim(radii_hi, flux_hi, radii_ha, flux_ha, radii_irac, flux_irac, axs, galaxy, index)
    
                #save plots
                #pdf.savefig()

            else:
                #hide blank axis
                ax1 = axs[3]
                ax1.set_axis_off()

        else:
            #hide blank axis
            ax1 = axs[3]
            ax1.set_axis_off()
        
        #show cutouts
        plt.show()
        
        #close fig
        plt.close()